# Poisson-Disk Sampling

**Domain:** Procedural Generation  ·  *recommended addition*  ·  **runnable:** yes

A refresher on generating point sets where no two points are closer than a radius `r`,
yet the points still look random — the *blue-noise* distribution that underlies good
object scattering, sampling, and stippling.

## 1. What & Why

**What it is.** Poisson-disk sampling produces a set of points that are *maximally random
subject to a minimum-distance constraint*: pick any two samples and they are at least a
radius `r` apart, but there is no grid, no lattice, no visible pattern. The frequency
spectrum of such a set is **blue noise** — energy concentrated at high frequencies, almost
none at low frequencies. That is exactly the property the eye finds "evenly spread but not
artificial."

**The problem it solves.** Two naive alternatives both fail:

- **Uniform random (`np.random.rand`)** — points clump and leave holes. Scatter 200 trees
  this way and you get bald patches next to dense thickets.
- **Regular grid / jitter** — perfectly even, but the eye instantly reads the lattice, and
  aliasing artifacts show up in rendering and sampling.

Poisson-disk sits in between: even coverage *without* a detectable pattern.

**When to reach for it.**

- Scattering objects in PCG worlds — trees, rocks, grass, enemies, loot — so nothing
  overlaps and density looks natural.
- **Anti-aliasing / Monte-Carlo sampling** in renderers; blue-noise sample sets converge
  with far less visible noise than white noise.
- Stippling, halftoning, and dithering (blue-noise dither masks).
- Procedural placement of cities/POIs on a map, dart-throwing for mesh generation.

**When not to.** If you need *exactly N* points, this is awkward (you control density via
`r`, not count). If overlap is fine or you want clustering, plain random is simpler and
faster. If you need a strict lattice (tilemaps), use a grid.

## 2. Mental Model

**Throwing non-overlapping coins on a table.** Each sample is a coin of radius `r/2`. You
keep tossing coins, but reject any toss that would overlap a coin already down. When you
can't fit any more, you stop. The result covers the table evenly with gaps no larger than a
coin — but the coins aren't in rows.

Naive "dart throwing" does exactly this but gets *exponentially* slow as the table fills
(most darts land on existing coins). **Bridson's algorithm** is the trick that keeps it
fast: maintain an *active list* of recently-placed points and only throw new darts in the
annulus `r..2r` around an active point. A **background grid** with cell size `r/√2`
guarantees at most one point per cell, so the "does this overlap anything?" test only checks
a handful of neighboring cells instead of every point. That makes the whole thing **O(N)**.

## 3. Key Concepts

- **Minimum radius `r`** — the only knob for density. Smaller `r` → more, tighter points.
  Expected count in 2D scales roughly as `area / r²`.
- **Blue noise** — the spectral signature: no low-frequency energy, so no clumps or holes.
  This is *why* the result looks good, not just a side effect.
- **Active list** — points still worth sampling around. A point leaves the list after `k`
  failed attempts to spawn a neighbor near it.
- **`k` (attempts per point)** — how many candidate darts to throw around each active point
  before giving up on it. Typically 30. Higher `k` packs slightly tighter at more cost.
- **Background grid, cell = `r/√2`** — the diagonal of a cell is `r`, so a cell can hold at
  most one sample. Neighbor checks become O(1).
- **Annulus `r` to `2r`** — candidates are spawned in this ring around an active point.
  Inner bound `r` enforces the constraint; outer bound `2r` keeps coverage tight (no big
  gaps). Sampling beyond `2r` would leave holes.
- **Dart throwing** — the original O(N²)-ish rejection method; correct but slow as it fills.
  Bridson is the standard fast replacement.

## 4. Setup

Pure-Python + NumPy is enough for the algorithm; Matplotlib is only for the visualization.
No GPU, no downloads, no API keys.

```bash
pip install numpy matplotlib
```

In [1]:
# %pip install numpy matplotlib
import math
import numpy as np

rng = np.random.default_rng(7)  # fixed seed -> reproducible output
print("numpy", np.__version__)

numpy 2.4.6


## 5. Worked Examples

### Example 1 — Bridson's fast Poisson-disk sampling (2D)

The canonical O(N) algorithm (Bridson, SIGGRAPH 2007). We sample a `width × height` region
with minimum spacing `r`, using a background grid for fast neighbor lookups and an active
list to drive growth.

In [2]:
def poisson_disk(width, height, r, k=30, rng=rng):
    """Bridson's algorithm. Returns an (N, 2) array of points >= r apart."""
    cell = r / math.sqrt(2)                       # at most one point per grid cell
    gw, gh = int(math.ceil(width / cell)), int(math.ceil(height / cell))
    grid = -np.ones((gw, gh), dtype=int)          # cell -> index into samples, or -1
    samples, active = [], []

    def grid_coords(p):
        return int(p[0] / cell), int(p[1] / cell)

    def fits(p):
        if not (0 <= p[0] < width and 0 <= p[1] < height):
            return False
        gx, gy = grid_coords(p)
        for ix in range(max(gx - 2, 0), min(gx + 3, gw)):
            for iy in range(max(gy - 2, 0), min(gy + 3, gh)):
                j = grid[ix, iy]
                if j != -1 and np.hypot(*(p - samples[j])) < r:
                    return False
        return True

    # seed with one random point
    p0 = np.array([rng.uniform(0, width), rng.uniform(0, height)])
    samples.append(p0); active.append(0); grid[grid_coords(p0)] = 0

    while active:
        i = active[rng.integers(len(active))]
        base = samples[i]
        for _ in range(k):                        # try k candidates in the annulus r..2r
            ang = rng.uniform(0, 2 * math.pi)
            rad = rng.uniform(r, 2 * r)
            cand = base + rad * np.array([math.cos(ang), math.sin(ang)])
            if fits(cand):
                idx = len(samples)
                samples.append(cand); active.append(idx)
                grid[grid_coords(cand)] = idx
                break
        else:                                     # k failures -> retire this point
            active.remove(i)

    return np.array(samples)


pts = poisson_disk(width=20, height=20, r=1.0)
print(f"placed {len(pts)} points in a 20x20 area with r=1.0")
print("first 3 points:\n", np.round(pts[:3], 3))

placed 264 points in a 20x20 area with r=1.0
first 3 points:
 [[12.502 17.944]
 [12.699 16.735]
 [11.225 18.205]]


### Verify the constraint and compare against uniform random

The whole promise is "no two points closer than `r`." Let's check the minimum pairwise
distance for the Poisson-disk set, then show that plain uniform random violates it badly.

In [3]:
from scipy.spatial.distance import pdist  # falls back below if scipy missing

def min_pairwise(points):
    try:
        return pdist(points).min()
    except Exception:
        d = np.inf
        for i in range(len(points)):
            diff = points[i + 1:] - points[i]
            if len(diff):
                d = min(d, np.hypot(diff[:, 0], diff[:, 1]).min())
        return d

uniform = rng.uniform(0, 20, size=(len(pts), 2))   # same count, plain random

print(f"Poisson-disk  min distance = {min_pairwise(pts):.3f}   (target r = 1.0)")
print(f"Uniform random min distance = {min_pairwise(uniform):.3f}   (often << r -> clumps)")

Poisson-disk  min distance = 1.001   (target r = 1.0)
Uniform random min distance = 0.037   (often << r -> clumps)


### Example 2 — see the difference (blue noise vs white noise)

Side by side, Poisson-disk points are evenly spread with consistent gaps; uniform random
shows clumps and voids. This is the visible payoff of blue noise.

In [4]:
import matplotlib
matplotlib.use("Agg")  # headless-safe; no display needed
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 4.5))
ax1.scatter(pts[:, 0], pts[:, 1], s=12, c="#1f77b4")
ax1.set_title(f"Poisson-disk (blue noise) — {len(pts)} pts")
ax2.scatter(uniform[:, 0], uniform[:, 1], s=12, c="#d62728")
ax2.set_title(f"Uniform random (white noise) — {len(uniform)} pts")
for ax in (ax1, ax2):
    ax.set_xlim(0, 20); ax.set_ylim(0, 20); ax.set_aspect("equal")
plt.tight_layout()
plt.savefig("poisson_vs_uniform.png", dpi=80)
print("saved poisson_vs_uniform.png — left is evenly spread, right clumps")
plt.show()

saved poisson_vs_uniform.png — left is evenly spread, right clumps


/var/folders/p8/sm5jmh055md_zzhhn1mfgyw80000gn/T/ipykernel_43297/3942604164.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Gotchas & Pitfalls

- **You don't get a fixed point count.** Density is set by `r`; the final `N` falls out of
  the geometry. If you need exactly N points, sample more than enough and truncate, or
  binary-search `r`.
- **Cell size must be `r/√2`, not `r`.** With cell = `r` a single cell could hold two points
  closer than `r`, breaking the O(1) neighbor check. The `/√2` makes the cell diagonal equal
  `r`, guaranteeing one point per cell.
- **Check a 5×5 neighborhood, not 3×3.** A valid neighbor can sit up to `2r` away (the
  annulus outer edge), which spans two cells. Checking only the immediate ring misses
  conflicts and lets points slip in too close. (We scan `gx-2..gx+2`.)
- **Naive dart throwing dies on dense sets.** Without the active list + grid, rejection
  sampling is fine at low density but grinds to a halt as the domain fills — most darts hit
  existing points. Always use Bridson for non-trivial counts.
- **Boundaries / tiling.** Points respect the constraint *inside* the domain but not across
  edges. For seamless tiling worlds, wrap coordinates toroidally in the distance check, or
  generate with periodic boundaries.
- **Variable-radius / weighted sampling needs care.** If `r` varies in space (denser forest
  here, sparse there), the grid cell size must use the *smallest* `r`, and the fits-check
  must use the local `r` — easy to get subtly wrong.
- **High dimensions are expensive.** Bridson generalizes to nD, but the annulus volume and
  grid neighborhood grow fast; beyond ~4D it's rarely the right tool.

## 7. When to Use vs Alternatives

| Method | Distribution | Even coverage? | Visible pattern? | Cost | Use when |
|---|---|---|---|---|---|
| **Poisson-disk (Bridson)** | Blue noise | Yes | No | O(N) | Natural scattering, AA sampling, stippling |
| **Uniform random** | White noise | No (clumps/holes) | No | O(N) | Overlap OK, or you *want* clustering |
| **Regular grid** | — | Perfect | Yes (lattice) | O(N) | Tilemaps, when a pattern is acceptable |
| **Jittered grid** | Near-blue | Yes-ish | Faint grid | O(N) | Quick approximation, sampling |
| **Lloyd relaxation (CVT)** | Very even | Very (too even) | Slightly | O(N·iters) | Cell/Voronoi layouts, when ultra-uniform is wanted — see [[voronoi-delaunay]] |
| **Halton / Sobol (QMC)** | Low-discrepancy | Yes | Subtle | O(N) | Deterministic integration sampling |

**Rules of thumb.**
- Want it to "look natural and evenly spread"? → **Poisson-disk**.
- Want *too-perfect* even spacing (e.g. relaxed Voronoi cells)? → **Lloyd relaxation** on top
  of a random or Poisson start.
- Doing Monte-Carlo integration and want determinism? → **Halton/Sobol**.
- Jittered grid is the cheap 80%-solution if you don't need a hard minimum distance.

Poisson-disk pairs naturally with other PCG tools: place biome anchors with Poisson-disk,
then grow regions with [[voronoi-delaunay]] or scatter detail with [[perlin-noise]].

## 8. Resources

- **Bridson, "Fast Poisson Disk Sampling in Arbitrary Dimensions" (SIGGRAPH 2007 sketch)** —
  the two-page paper this notebook implements:
  https://www.cs.ubc.ca/~rbridson/docs/bridson-siggraph07-poissondisk.pdf
- **Mike Bostock, "Visualizing Algorithms"** — beautiful animated walk-through of Bridson's
  algorithm and why blue noise matters: https://bost.ocks.org/mike/algorithms/
- **Herman Tulleken, "Poisson Disk Sampling" (Dev.Mag)** — practical PCG-oriented tutorial
  with code: http://devmag.org.za/2009/05/03/poisson-disk-sampling/
- **Robert Bridson & others, blue-noise overview (A. Wolfe / "The problem with 3D Blue
  Noise")** — deeper dive on the spectrum and sampling applications:
  https://blog.demofox.org/2017/10/20/generating-blue-noise-sample-points-with-mitchells-best-candidate-algorithm/
- **`scipy.stats.qmc.PoissonDisk`** — SciPy's built-in implementation if you'd rather not
  hand-roll it: https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.qmc.PoissonDisk.html